# Associative inference on one H200: tasks 1 and 2

Two follow-ups to the associative-inference gate, where the 2B and 4B failed everywhere (the gate in `configs/assoc_inf_ab.yaml`: the direct probes A→B and B→C must be above chance with no answer bias, for both models, at 2 and 5 chains).

**Task 1. Was the answer read before the model had decided?** In the gate run about half the 2B/4B replies were reasoning cut off at 100 tokens, while the forced choice was read with no reasoning in context. Here the 2B and 4B get the same cells at 2 and 5 chains (same episodes, same prompt), but each trial lets the model reply first, up to 512 tokens, and then reads its 1-or-2 choice with that reply in context (`"<reply>\n\nAnswer: "`). Every trial is compared with its twin from the gate run.

**Task 2. Is it a scale problem?** The repo's gate, unchanged, on the 9B and 27B: the same 54 cells per model, 50 trials each, standard scoring.

The ablations (other prompts, more seeds) wait until these are in.

**What happens, top to bottom**

1. Check the GPU, driver, disk and RAM.
2. Unpack the repo, and restore the results zip from the earlier pod (it holds the 2B/4B gate run that task 1 compares against).
3. Build the pinned environment, download the stimuli and the 2B, 4B, 9B and 27B weights.
4. Write the configs and scripts into the repo (no existing repo file is modified).
5. Preflight: two trials per model at the largest cells, checked for errors, real answer probabilities and readable replies, with a time estimate.
6. The full runs in the background (task 1, then task 2). Closing the browser does not stop them.
7. Results: the gate for each task, the trial-for-trial comparison, figures, and one zip to download.

## Which pod to rent

**One H200** (141 GB). The 27B needs ~65 GB here and everything else less, so an 80 GB A100 or H100 also works, just slower. Choose **CUDA 12.8 or newer**, a **volume of 150 GB or more at `/workspace`**, 16 GB+ RAM, and on-demand rather than spot.

**Before you start, upload to `/workspace`** (drag into the Jupyter file browser):
- `qwen35-bench-master.zip` (the repo), or set `GIT_URL` below
- the results zip from the earlier pod, `qwen35_runpod_results_20261012_0106.zip`. Task 1 compares against the 2B/4B gate results inside it. Without it, the queue first re-runs that gate (about an hour more).

**Time:** roughly 6–12 hours in total, mostly the 27B; the preflight prints a better estimate for this pod. Every cell is safe to re-run, and finished results are skipped, so a stopped run resumes where it stopped.

**Run All** works: the long jobs run in the background and the `wait(...)` cells block until they finish. A model whose preflight fails is held back automatically.

## 1. Settings

In [ ]:
import os
from pathlib import Path

WORK = Path(os.environ.get("BENCH_WORK", "/workspace"))   # the pod's persistent volume
REPO = WORK / "qwen35-bench"                      # the repo is unpacked here
REPO_ZIP = WORK / "qwen35-bench-master.zip"       # upload the zip here...
GIT_URL = ""                                      # ...or clone instead: "https://github.com/<owner>/<repo>.git"
GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN", "") # only for cloning a private repo; never paste a token here
HF_TOKEN = os.environ.get("HF_TOKEN", "")         # optional: the Qwen3.5 checkpoints are public

RUN_TASK1 = True         # 2B and 4B, reason-then-choose scoring (configs/runpod/assoc_inf_reasoned.yaml)
RUN_TASK2 = True         # 9B and 27B, the repo's gate unchanged (configs/runpod/assoc_inf_gate_big.yaml)
GPUS = "0"               # GPU(s) for the jobs; "0,1" would shard the 27B over two cards (not needed on an H200)
CPU_THREADS = 0          # 0 = read the pod's real CPU limit (too many threads makes jobs several times slower)
FORCE_MAIN = False       # True launches a model even if its preflight failed (not recommended)
PRICE_PER_HOUR = 0.0     # your pod's $/hour, for the cost estimate (optional)

VENV = WORK / "envs" / "qwen35"
PY = VENV / "bin" / "python"
DATA = WORK / "memory_datasets"
HF_HOME = WORK / "hf_cache"
LOGS = WORK / "runpod_logs"
OUT = WORK / "outputs"

Helpers used by the cells below. Nothing to edit; run once per kernel session (and again after a kernel restart).

In [ ]:
import json, re, shutil, signal, subprocess, sys, time, urllib.request, zipfile
from datetime import datetime
from IPython.display import Image, Markdown, clear_output, display

ENV = dict(os.environ, HF_HOME=str(HF_HOME), PYTHONUNBUFFERED="1",
           TRITON_CACHE_DIR=str(WORK / ".triton"), UV_CACHE_DIR=str(WORK / ".uv_cache"),
           UV_PYTHON_INSTALL_DIR=str(WORK / ".uv_python"))
if HF_TOKEN:
    ENV["HF_TOKEN"] = HF_TOKEN
RUN_ENV = {"HF_HUB_OFFLINE": "1"}   # the benchmark jobs read weights from the local cache only
for d in (LOGS, OUT, HF_HOME):
    d.mkdir(parents=True, exist_ok=True)


def cpu_quota():
    """Cores this pod may actually use (its cgroup CPU limit), or None if there is no limit."""
    try:
        quota, period = Path("/sys/fs/cgroup/cpu.max").read_text().split()[:2]
    except (OSError, ValueError):
        try:
            quota = Path("/sys/fs/cgroup/cpu/cpu.cfs_quota_us").read_text().strip()
            period = Path("/sys/fs/cgroup/cpu/cpu.cfs_period_us").read_text().strip()
        except OSError:
            return None
    if quota == "max" or int(quota) <= 0:
        return None
    return int(quota) / int(period)


VISIBLE_CPUS = os.cpu_count() or 1
CPU_QUOTA = cpu_quota()
THREADS = CPU_THREADS or max(1, int(min(VISIBLE_CPUS, CPU_QUOTA or VISIBLE_CPUS)))
RUN_ENV.update(OMP_NUM_THREADS=str(THREADS), MKL_NUM_THREADS=str(THREADS))
print(f"CPU: {VISIBLE_CPUS} cores visible, " + (f"{CPU_QUOTA:.1f} usable" if CPU_QUOTA else "no limit")
      + f"; benchmark jobs will use {THREADS} threads")

# ---- what runs ------------------------------------------------------------------------------------
EXT = "runpod/run_bench_ext.py"                            # run_bench.py plus reason-then-choose scoring
PLAIN_2B_4B = "configs/assoc_inf_ab.yaml"                  # the repo's gate: task 1's comparison run
TASK1 = "configs/runpod/assoc_inf_reasoned.yaml"
TASK2 = "configs/runpod/assoc_inf_gate_big.yaml"
PREFLIGHT1 = "configs/runpod/preflight_assoc_reasoned.yaml"
PREFLIGHT2 = "configs/runpod/preflight_assoc_big.yaml"
TASK1_MODELS = ("qwen3.5-2b", "qwen3.5-4b")
TASK2_MODELS = ("qwen3.5-9b", "qwen3.5-27b")
SIZES_GB = {"Qwen/Qwen3.5-2B": 5, "Qwen/Qwen3.5-4B": 9, "Qwen/Qwen3.5-9B": 19, "Qwen/Qwen3.5-27B": 55}


def models_needed():
    return (["Qwen/Qwen3.5-2B", "Qwen/Qwen3.5-4B"] if RUN_TASK1 else []) + \
           (["Qwen/Qwen3.5-9B", "Qwen/Qwen3.5-27B"] if RUN_TASK2 else [])


def gpu_memory():
    """{GPU index: GB}, decimal GB like the result files' peak_gpu_gb. Empty without nvidia-smi."""
    smi = shutil.which("nvidia-smi")
    if not smi:
        return {}
    out = subprocess.run([smi, "--query-gpu=index,memory.total", "--format=csv,noheader,nounits"],
                         capture_output=True, text=True).stdout
    mem = {}
    for line in out.strip().splitlines():
        try:
            index, mib = [x.strip() for x in line.split(",")]
            mem[int(index)] = float(mib) * 2**20 / 1e9
        except ValueError:
            pass
    return mem


def capacity(gpus=None):
    mem = gpu_memory()
    return sum(mem.get(int(i), 0) for i in str(gpus or GPUS).split(","))


def run(cmd, cwd=None, check=True, echo=True):
    """Run a command, streaming its output into the cell. Returns the exit code if non-zero, else None."""
    shell = isinstance(cmd, str)
    if not shell:
        cmd = [str(c) for c in cmd]
    if echo:
        print("$", cmd if shell else " ".join(cmd), flush=True)
    p = subprocess.Popen(cmd, shell=shell, cwd=cwd, env=ENV, text=True, bufsize=1,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in p.stdout:
        print(line, end="", flush=True)
    rc = p.wait()
    if check and rc != 0:
        raise RuntimeError(f"command failed with exit code {rc}")
    return rc or None


def bench(*args, check=True):
    """Run the benchmark environment's Python from the repo root."""
    return run([PY, *args], cwd=REPO, check=check, echo=False)


def gpu_usage():
    smi = shutil.which("nvidia-smi")
    if not smi:
        return
    out = subprocess.run([smi, "--query-gpu=index,utilization.gpu,memory.used,memory.total,power.draw",
                          "--format=csv,noheader,nounits"], capture_output=True, text=True).stdout
    for line in out.strip().splitlines():
        fields = [x.strip() for x in line.split(",")]
        if len(fields) < 4:
            continue
        i, util, used, total = fields[:4]
        power = f", {fields[4]} W" if len(fields) > 4 else ""
        print(f"GPU {i}: {util}% busy, {float(used) / 1024:.0f} of {float(total) / 1024:.0f} GiB in use{power}")


# ---- background job queues --------------------------------------------------------------------
_QUEUES = {}


def _proc_fields(pid):
    try:
        return Path(f"/proc/{pid}/stat").read_text().rsplit(")", 1)[1].split()[:3]
    except (OSError, IndexError):
        return None


def queue_running(name):
    p = _QUEUES.get(name)
    if p is not None:
        return p.poll() is None
    pid_file = LOGS / f"{name}.pid"
    if not pid_file.exists():
        return False
    pid = int(pid_file.read_text())
    fields = _proc_fields(pid)
    if fields is None or fields[0] in ("Z", "X"):
        return False
    try:
        return b"run_queue.py" in Path(f"/proc/{pid}/cmdline").read_bytes()
    except OSError:
        return False


def launch(name, jobs):
    """Run jobs one after another in the background. Each job is (config, gpus, entry script or None),
    optionally with a 4th element: a list of model names to run (default: every model in the config)."""
    jobs = [tuple(j) + (None,) * (4 - len(j)) for j in jobs]
    if queue_running(name):
        print(f"Queue '{name}' is already running; watch it with status('{name}').")
        return
    for config, *_ in jobs:
        if not (REPO / config).exists():
            raise FileNotFoundError(f"{config} is not in {REPO}")
    queue_file = LOGS / f"{name}.queue.json"
    queue_file.write_text(json.dumps({
        "name": name, "python": str(PY), "repo": str(REPO), "log_dir": str(LOGS), "env": RUN_ENV,
        "jobs": [{"config": c, "task": None, "gpus": str(g), "entry": e,
                  "extra_args": [a for m in (models or []) for a in ("--model", m)]}
                 for c, g, e, models in jobs]}, indent=2))
    (LOGS / f"{name}.status.json").unlink(missing_ok=True)
    log = open(LOGS / f"{name}.queue.log", "a")
    p = subprocess.Popen([str(PY), "runpod/run_queue.py", str(queue_file)], cwd=REPO, env=ENV,
                         stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
    _QUEUES[name] = p
    (LOGS / f"{name}.pid").write_text(str(p.pid))
    print(f"Started queue '{name}' (pid {p.pid}). Jobs, in order:")
    for config, gpus, entry, models in jobs:
        print(f"  {entry or 'run_bench.py'} {config}   GPU {gpus}" + (f"   only {', '.join(models)}" if models else ""))
    print(f"Logs: {LOGS}")


def status(name="main", tail=6):
    """Job states, cells done, the running job's last log lines, GPU use."""
    st_file = LOGS / f"{name}.status.json"
    running = queue_running(name)
    if not st_file.exists():
        print(f"Queue '{name}': " + ("starting..." if running else "never launched."))
        return
    st = json.loads(st_file.read_text())
    state = ("running" if running else "finished" if st.get("finished")
             else "stopped" if st.get("stopped") else "not running")
    print(f"Queue '{name}': {state}   started {st['started']}"
          + (f", finished {st['finished']}" if st.get("finished") else ""))
    for job in st["jobs"]:
        took = ""
        if job.get("start"):
            end = datetime.fromisoformat(job["end"]) if job.get("end") else datetime.now()
            took = str(end - datetime.fromisoformat(job["start"])).split(".")[0]
        print(f"  [{job['state']:8s}] {job['config']}   GPU {job['gpus']}   {took}")
    print()
    bench("runpod/progress.py", *[j["config"] for j in st["jobs"]], check=False)
    active = [j for j in st["jobs"] if j["state"] == "running"]
    if active and active[0].get("log") and Path(active[0]["log"]).exists():
        print(f"\nLast lines of {Path(active[0]['log']).name}:")
        for line in Path(active[0]["log"]).read_text(errors="replace").splitlines()[-tail:]:
            print("   ", line[:220])
        if not running:
            print("\nThe queue process is gone but a job is still marked running (pod restart, or stop()).\n"
                  "Launch the same queue again: finished cells are kept and skipped.")
    for j in st["jobs"]:
        if j["state"] == "failed":
            print(f"\nFAILED: {j['config']} (exit code {j['returncode']}). End of its log, {Path(j['log']).name}:")
            for line in Path(j["log"]).read_text(errors="replace").splitlines()[-8:]:
                print("   ", line[:220])
    gpu_usage()


def wait(name="main", every=60):
    """Refresh status(name) until the queue finishes. Interrupting only stops watching; the jobs keep running."""
    try:
        while queue_running(name):
            clear_output(wait=True)
            status(name)
            print(f"\nRefreshing every {every} s. Interrupt this cell any time; the jobs keep running.")
            time.sleep(every)
    except KeyboardInterrupt:
        print("Stopped watching. The jobs keep running; check them with status().")
        return
    clear_output(wait=True)
    status(name)


def _group_alive(pgid):
    for stat in Path("/proc").glob("[0-9]*/stat"):
        try:
            state, _, group = stat.read_text().rsplit(")", 1)[1].split()[:3]
        except (OSError, IndexError, ValueError):
            continue
        if group == str(pgid) and state not in ("Z", "X"):
            return True
    return False


def stop(name="main"):
    """Stop a queue and the job it is running. Finished cells are kept; launching again resumes."""
    if not queue_running(name):
        print(f"Queue '{name}' is not running.")
        return
    pgid = int((LOGS / f"{name}.pid").read_text())
    os.killpg(pgid, signal.SIGTERM)
    for _ in range(40):
        if not _group_alive(pgid):
            break
        time.sleep(0.5)
    else:
        try:
            os.killpg(pgid, signal.SIGKILL)
        except ProcessLookupError:
            pass
    if name in _QUEUES:
        _QUEUES.pop(name).poll()
    st_file = LOGS / f"{name}.status.json"
    if st_file.exists():
        st = json.loads(st_file.read_text())
        now = datetime.now().isoformat(timespec="seconds")
        for j in st["jobs"]:
            if j["state"] == "running":
                j.update(state="stopped", end=now)
        st["stopped"] = now
        st_file.write_text(json.dumps(st, indent=2))
    print(f"Stopped queue '{name}'.")


def progress(config):
    """{done, failed, skipped, todo, total} for one config, without loading a model."""
    r = subprocess.run([str(PY), "runpod/progress.py", "--json", config], cwd=REPO, env=ENV,
                       capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout + r.stderr)
        return None
    return json.loads(r.stdout)[0]


def show_replies(run_dir, models=None, k=2):
    """A few raw replies per model, probe and pair type, from the largest cell, to eyeball for gibberish."""
    cells = {}
    for f in sorted((REPO / run_dir).glob("*/*.json")):
        d = json.loads(f.read_text())
        c, model = d["run"]["condition"], d["run"]["model"]["name"]
        if models and model not in models:
            continue
        key = (model, c.get("pair_type", ""), c.get("probe", ""))
        if key not in cells or c.get("n_images", 0) > cells[key][0]:
            cells[key] = (c.get("n_images", 0), d)
    for key, (n, d) in sorted(cells.items()):
        c = d["run"]["condition"]
        error = f"   ERROR: {d['error'].strip().splitlines()[-1][:150]}" if d.get("error") else ""
        print(f"\n{' '.join(key)}   {c.get('n_chains')} chains, {c.get('prompt')}{error}")
        for t in (d.get("trials") or [])[:k]:
            print(f"   target={t.get('target')} choice={t.get('reported')} mass={t.get('choice_mass', 0):.3f} "
                  f"reply={(t.get('free_response') or '')[:160]!r}")


print("Helpers ready.")

## 2. GPU, driver, disk and RAM

In [ ]:
smi = shutil.which("nvidia-smi")
if smi is None:
    raise RuntimeError("nvidia-smi not found: this pod has no visible GPU.")
out = subprocess.run([smi, "--query-gpu=index,name,memory.total,driver_version", "--format=csv,noheader,nounits"],
                     capture_output=True, text=True, check=True).stdout
GPU_MEM, driver = {}, "0"
for line in out.strip().splitlines():
    idx, name, mem, driver = [x.strip() for x in line.split(",")]
    GPU_MEM[int(idx)] = float(mem) * 2**20 / 1e9
    print(f"GPU {idx}: {name}, {float(mem) / 1024:.0f} GiB = {GPU_MEM[int(idx)]:.1f} GB, driver {driver}")

fatal, warnings = [], []
if int(driver.split(".")[0]) < 570:
    fatal.append(f"Driver {driver} is older than 570; torch 2.10 with CUDA 12.8 needs 570+. "
                 "Deploy a new pod filtered to CUDA 12.8 or newer.")
missing = [g for g in GPUS.split(",") if int(g) not in GPU_MEM]
if missing:
    fatal.append(f"GPU {','.join(missing)} is in the settings but not on this pod.")
if RUN_TASK2 and capacity() < 75:
    warnings.append(f"The 27B needs ~65 GB here; GPUS={GPUS!r} gives {capacity():.0f} GB. The preflight will show "
                    "whether it fits.")

pending = [m for m in models_needed() if not (HF_HOME / "hub" / ("models--" + m.replace("/", "--"))).exists()]
need = sum(SIZES_GB[m] for m in pending) + (0 if PY.exists() else 12) + 5
free = shutil.disk_usage(WORK).free / 1e9
print(f"\nDisk: {free:.0f} GB free in {WORK}; about {need:.0f} GB still needed (weights, environment, headroom)")
if free < need:
    fatal.append(f"Not enough disk in {WORK}: {free:.0f} GB free, about {need:.0f} GB needed. Use a 150 GB volume.")

ram = int(re.search(r"MemTotal:\s+(\d+)", Path("/proc/meminfo").read_text()).group(1)) / 1e6
for limit_file in ("/sys/fs/cgroup/memory.max", "/sys/fs/cgroup/memory/memory.limit_in_bytes"):
    try:
        value = Path(limit_file).read_text().strip()
        if value.isdigit():
            ram = min(ram, int(value) / 1e9)
    except OSError:
        pass
print(f"RAM: {ram:.0f} GB")
if ram < 16:
    warnings.append(f"Only {ram:.0f} GB of RAM; 16 GB+ is safer.")

print()
for w in warnings:
    print("WARNING:", w)
if fatal:
    raise RuntimeError("Fix before continuing:\n  - " + "\n  - ".join(fatal))
print("Checks passed." if not warnings else "No blocking problems.")

## 3. Get the repo, and restore the earlier results

In [ ]:
if (REPO / "run_bench.py").exists():
    print(f"Repo already at {REPO}")
elif GIT_URL:
    url = GIT_URL
    if GITHUB_TOKEN and url.startswith("https://"):
        url = url.replace("https://", f"https://x-access-token:{GITHUB_TOKEN}@", 1)
    print(f"Cloning {GIT_URL} ...")
    run(["git", "clone", "--depth", "1", url, REPO], echo=False)     # not echoed: the URL may hold a token
elif REPO_ZIP.exists():
    with zipfile.ZipFile(REPO_ZIP) as z:
        top = z.namelist()[0].split("/")[0]
        z.extractall(WORK)
        commit = z.comment.decode(errors="replace").strip()
    (WORK / top).rename(REPO)
    (REPO / "RUNPOD_SOURCE.json").write_text(json.dumps(
        {"source": REPO_ZIP.name, "archive_commit": commit or None,
         "unpacked": datetime.now().isoformat(timespec="seconds")}, indent=2))
else:
    raise FileNotFoundError(f"No repo yet. Upload qwen35-bench-master.zip to {WORK} "
                            "(drag it into the Jupyter file browser), or set GIT_URL in the settings cell.")

for p in ("run_bench.py", "bench/runner.py", "bench/models.py", "bench/tasks/associative_inference_eval.py",
          PLAIN_2B_4B, "requirements.txt", "requirements.lock.txt"):
    assert (REPO / p).exists(), f"{p} is missing from the repo"
if (REPO / "RUNPOD_SOURCE.json").exists():
    print("Unpacked from", json.loads((REPO / "RUNPOD_SOURCE.json").read_text()))

Copies the finished result files from any results zip in `/workspace` (`qwen35_runpod_results_*.zip`, from the earlier pod or from an earlier run of this notebook) into the repo, so those cells are skipped. Files already in the repo are never overwritten.

In [ ]:
previous = sorted(WORK.glob("qwen35_runpod_results_*.zip"))
for zip_path in previous:
    restored = kept = 0
    try:
        with zipfile.ZipFile(zip_path) as z:
            for name in z.namelist():
                if not (name.startswith("results/") and name.endswith(".json")):
                    continue
                if (REPO / name).exists():
                    kept += 1
                else:
                    z.extract(name, REPO)
                    restored += 1
    except zipfile.BadZipFile:
        print(f"{zip_path.name} is not a complete zip (the upload may have been cut off); upload it again.")
        continue
    print(f"{zip_path.name}: restored {restored} result files" + (f", kept {kept} already here" if kept else ""))
if not previous:
    print(f"No results zip in {WORK}.")

twins = len([f for f in (REPO / "results/assoc_inf_ab/associative_inference").glob("*.json")
             if f.name.startswith(TASK1_MODELS)]) if (REPO / "results/assoc_inf_ab").exists() else 0
print(f"\n2B/4B gate results (task 1's comparison): {twins} of 108 cells here.")
if RUN_TASK1 and twins < 108:
    print("The missing ones are re-run first in the main queue (about an hour for all of them). To skip that, "
          "upload qwen35_runpod_results_20261012_0106.zip to /workspace and run this cell again.")

## 4. Python environment

The pins of the repo's `setup_env.sh` and `requirements.lock.txt`, in its own Python 3.12 environment on the volume. The first build takes 5–10 minutes; afterwards this cell only checks it.

In [ ]:
CAUSAL_CONV1D_WHEEL = ("https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.7.0/"
                       "causal_conv1d-1.7.0+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl")
CHECK = ("import torch, transformers, fla, causal_conv1d; assert torch.cuda.is_available(), 'CUDA not available'; "
         "print('torch', torch.__version__, '| transformers', transformers.__version__, "
         "'| GPU', torch.cuda.get_device_name(0))")


def env_ok():
    if not PY.exists():
        return False
    r = subprocess.run([str(PY), "-c", CHECK], env=ENV, capture_output=True, text=True)
    print((r.stdout + r.stderr).strip()[-1500:])
    return r.returncode == 0


if env_ok():
    print("Environment ready:", VENV)
else:
    run([sys.executable, "-m", "pip", "install", "-q", "uv"])
    uv_bin = Path(sys.executable).parent / "uv"
    uv = [str(uv_bin)] if uv_bin.exists() else [sys.executable, "-m", "uv"]
    run([*uv, "venv", "--python", "3.12", VENV])
    run([*uv, "pip", "install", "--python", PY, "torch==2.10.0", "torchvision==0.25.0",
         "--index-url", "https://download.pytorch.org/whl/cu128"])
    run([*uv, "pip", "install", "--python", PY, CAUSAL_CONV1D_WHEEL])
    lock = (REPO / "requirements.lock.txt").read_text().splitlines()
    pins = [l for l in lock if l.strip() and not re.match(r"(torch|torchvision|triton|nvidia-|cuda-|causal-conv1d)\b", l)]
    (WORK / "constraints.txt").write_text("\n".join(pins) + "\n")
    run([*uv, "pip", "install", "--python", PY, "-r", REPO / "requirements.txt", "-c", WORK / "constraints.txt"])
    assert env_ok(), "The environment check failed; read the output above."
    print("Environment ready:", VENV)

## 5. Stimuli

From the public `csiyer/vision-memory` repo. Associative inference uses the Brady 2008 objects and the word pool; the MST sets come along in the same download and are checked too.

In [ ]:
STIMULI = {
    "memory_datasets_brady.zip": ("https://raw.githubusercontent.com/csiyer/vision-memory/main/memory_datasets_brady.zip", 80955797),
    "memory_datasets_mst.zip": ("https://raw.githubusercontent.com/csiyer/vision-memory/main/memory_datasets_mst.zip", 63189250),
}
EXPECTED = {"Brady2008Objects": 2400, "Brady2008Exemplar": 200, "Brady2008State": 200,
            "Brady2013ColorObjects": 540, **{f"MST/Set {i}": 384 for i in range(1, 7)}}


def download(url, target, size, attempts=4):
    part = target.with_name(target.name + ".part")
    for attempt in range(1, attempts + 1):
        try:
            urllib.request.urlretrieve(url, part)
        except OSError as e:
            print(f"  attempt {attempt} of {attempts} failed ({e.__class__.__name__}); retrying", flush=True)
            time.sleep(5 * attempt)
            continue
        if part.stat().st_size != size:
            raise RuntimeError(f"{target.name} is {part.stat().st_size} bytes, expected {size}: "
                               "the source repo may have changed, so check the stimuli before running.")
        part.replace(target)
        return
    raise RuntimeError(f"Could not download {url} after {attempts} attempts.")


def count_images(folder):
    if not folder.is_dir():
        return 0
    return sum(1 for p in folder.iterdir() if p.suffix.lower() in (".jpg", ".jpeg", ".png"))


def stimuli_report():
    counts = {k: count_images(DATA / k) for k in EXPECTED}
    pool = DATA / "wasnorm_wordpool.txt"
    words = len(pool.read_text().split()) if pool.exists() else 0
    bins = len(list((DATA / "MST").glob("Set* bins.txt"))) if (DATA / "MST").exists() else 0
    ok = all(counts[k] == v for k, v in EXPECTED.items()) and words >= 1637 and bins == 6
    return ok, counts, words, bins


ok, *_ = stimuli_report()
if not ok:
    downloads = WORK / "downloads"
    downloads.mkdir(exist_ok=True)
    for name, (url, size) in STIMULI.items():
        target = downloads / name
        if not (target.exists() and target.stat().st_size == size):
            print(f"Downloading {name} ({size / 1e6:.0f} MB) ...", flush=True)
            download(url, target, size)
        with zipfile.ZipFile(target) as z:
            z.extractall(WORK)

ok, counts, words, bins = stimuli_report()
for k, v in counts.items():
    print(f"  {k:24s} {v:5d} images (expected {EXPECTED[k]})")
print(f"  word pool: {words} words; MST similarity-bin files: {bins} of 6")
assert ok, "The stimuli are incomplete; see the counts above."

link = REPO / "memory_datasets"
if link.is_symlink() and link.resolve() != DATA.resolve():
    link.unlink()
if not link.exists() and not link.is_symlink():
    link.symlink_to(DATA)
print(f"\n{link} -> {link.resolve()}")

## 6. Model weights

About 5 GB (2B), 9 GB (4B), 19 GB (9B) and 55 GB (27B), cached in `/workspace/hf_cache`. Already-downloaded files are skipped.

In [ ]:
for repo_id in models_needed():
    print(f"\n== {repo_id}", flush=True)
    bench("-c", f"from huggingface_hub import snapshot_download; print(snapshot_download({repo_id!r}))")
print("\nWeights ready.")

## 7. Configs and scripts

Written into the repo under `configs/runpod/` and `runpod/`. No existing repo file is modified.

| File | What it is |
|---|---|
| `assoc_inf_reasoned.yaml` | task 1: the 2B/4B gate cells at 2 and 5 chains, 50 trials, seed 0, `scoring: reason_then_choose`, replies up to 512 tokens |
| `assoc_inf_gate_big.yaml` | task 2: `configs/assoc_inf_ab.yaml` with the 9B and 27B in place of the 2B and 4B; same run name, so the results sit beside the 2B/4B ones |
| `preflight_assoc_*.yaml` | two trials per model at the largest cells, under separate run names |
| `run_bench_ext.py` | `run_bench.py` plus the reason-then-choose evaluator. Models without the `scoring` key run exactly as under `run_bench.py` |
| `assoc_preflight.py` | the preflight checks and time estimate |
| `assoc_gate.py` | the repo's gate, for any pair of models |
| `assoc_reasoned_report.py` | task 1: every trial against its gate-run twin, with McNemar tests, reply lengths and cut-offs |
| `assoc_replies.py` | the "not gibberish" checks on the replies, per model and layout |
| `run_queue.py`, `progress.py`, `plot_extra.py` | background queue, progress counts, figures |

In [ ]:
(REPO / "configs" / "runpod").mkdir(parents=True, exist_ok=True)
(REPO / "runpod").mkdir(parents=True, exist_ok=True)
print("ok")

In [ ]:
%%writefile {REPO}/configs/runpod/assoc_inf_reasoned.yaml
# Task 1: does the 2B/4B associative-inference result change when the model may finish its reasoning?
#
# Same cells as configs/assoc_inf_ab.yaml at the gate sizes (2 and 5 chains), same seed, so every trial
# here sees exactly the episode and probe its assoc_inf_ab twin saw and the two compare trial for trial.
# One change: scoring: reason_then_choose (runpod/run_bench_ext.py). The model first replies freely, up
# to 512 tokens instead of 100, and the forced choice is then read with that reply in context
# ("<reply>\n\nAnswer: "). The prompt is unchanged.
#
# Why: in assoc_inf_ab about half the replies were reasoning cut off at 100 tokens, while the forced
# choice was read with no reasoning in context. The question: is the gate failure the models, or
# the answer being read before they had decided?
# Run with: python runpod/run_bench_ext.py configs/runpod/assoc_inf_reasoned.yaml

run_name: assoc_inf_reasoned

output_dir: results

seeds: [0]

models:
  - name: qwen3.5-2b
    model_id: Qwen/Qwen3.5-2B
    max_image_size: 512
    max_new_tokens: 512
    scoring: reason_then_choose
  - name: qwen3.5-4b
    model_id: Qwen/Qwen3.5-4B
    max_image_size: 512
    max_new_tokens: 512
    scoring: reason_then_choose

tasks:
  - task: associative_inference
    datasets: [Brady2008]
    pair_types: [image, word]
    probe_types: [inference, direct_ab, direct_bc]
    n_chains: [2, 5]
    prompts: [original, cue_turn, labeled]
    n_trials: 50


In [ ]:
%%writefile {REPO}/configs/runpod/assoc_inf_gate_big.yaml
# Task 2: the repo's associative-inference gate (configs/assoc_inf_ab.yaml) on the 9B and 27B.
#
# Identical to configs/assoc_inf_ab.yaml except for the models: same cells, layouts, probes, 50 trials,
# seed 0, standard scoring, max_new_tokens 100. Same run_name, so the results land beside the 2B/4B
# cells in results/assoc_inf_ab and summarize together. The gate is applied to the 9B and 27B as a pair,
# the same rule as for the 2B and 4B (runpod/assoc_gate.py --models qwen3.5-9b,qwen3.5-27b).
# The largest cells are 25 chains (100 images, about 7k tokens): small for both models on one H200.

run_name: assoc_inf_ab

output_dir: results

seeds: [0]

models:
  - name: qwen3.5-9b
    model_id: Qwen/Qwen3.5-9B
    max_image_size: 512
    max_new_tokens: 100
  - name: qwen3.5-27b
    model_id: Qwen/Qwen3.5-27B
    max_image_size: 512
    max_new_tokens: 100
    device_map: auto

tasks:
  - task: associative_inference
    datasets: [Brady2008]
    pair_types: [image, word]
    probe_types: [inference, direct_ab, direct_bc]
    n_chains: [2, 5, 25]
    prompts: [original, cue_turn, labeled]
    n_trials: 50


In [ ]:
%%writefile {REPO}/configs/runpod/preflight_assoc_reasoned.yaml
# Preflight for task 1: two trials per cell at the largest size it runs (5 chains), both pair types,
# one layout, with reason-then-choose scoring on the 2B and 4B. Separate run name, so these never pool
# with the real cells. Run with runpod/run_bench_ext.py.

run_name: runpod_preflight_assoc_reasoned

output_dir: results

seeds: [0]

models:
  - name: qwen3.5-2b
    model_id: Qwen/Qwen3.5-2B
    max_image_size: 512
    max_new_tokens: 512
    scoring: reason_then_choose
  - name: qwen3.5-4b
    model_id: Qwen/Qwen3.5-4B
    max_image_size: 512
    max_new_tokens: 512
    scoring: reason_then_choose

tasks:
  - task: associative_inference
    datasets: [Brady2008]
    pair_types: [image, word]
    probe_types: [inference]
    n_chains: [5]
    prompts: [original]
    n_trials: 2


In [ ]:
%%writefile {REPO}/configs/runpod/preflight_assoc_big.yaml
# Preflight for task 2: two trials per cell at the largest size (25 chains), both pair types, one layout,
# on the 9B and 27B. Separate run name, so these never pool with the real cells.

run_name: runpod_preflight_assoc

output_dir: results

seeds: [0]

models:
  - name: qwen3.5-9b
    model_id: Qwen/Qwen3.5-9B
    max_image_size: 512
    max_new_tokens: 100
  - name: qwen3.5-27b
    model_id: Qwen/Qwen3.5-27B
    max_image_size: 512
    max_new_tokens: 100
    device_map: auto

tasks:
  - task: associative_inference
    datasets: [Brady2008]
    pair_types: [image, word]
    probe_types: [direct_ab]
    n_chains: [25]
    prompts: [cue_turn]
    n_trials: 2


In [ ]:
%%writefile {REPO}/runpod/run_bench_ext.py
#!/usr/bin/env python
"""run_bench.py, plus one opt-in scoring mode: let the model reason, then read its choice.

Usage (from the repo root): python runpod/run_bench_ext.py <config.yaml> [run_bench.py's options]

A model entry with `scoring: reason_then_choose` gets an evaluator that, on every forced-choice
trial, first generates a free reply (greedy, up to the model's `max_new_tokens`, e.g. 512), and then
scores the choice with that reply in context: the prefill becomes "<reply>\n\nAnswer: " instead of
"Answer: ". The prompt is unchanged, so the only difference from the standard scoring is that the
model gets to finish whatever it wanted to say before its answer is read off the logits.

Why: at max_new_tokens 100 about half the 2B/4B associative-inference replies were reasoning that
was cut off before an answer, while the forced choice was read with no reasoning in context. This
separates "the model cannot do it" from "the model was cut off mid-thought".

The task code is untouched. The task calls score_choices(messages) and then _call_api(messages) on
the same messages; the second call returns the reply already generated (same greedy decode), so no
trial is generated twice. Per-trial reasoning stats (tokens used, whether the reply finished) are
added to each trial afterwards, and a few cell metrics with them. Models without the key behave
exactly as under run_bench.py, so one config can mix both.
"""
import re
import sys
from pathlib import Path

REPO_ROOT = Path(__file__).resolve().parent.parent
sys.path.insert(0, str(REPO_ROOT))

ANSWER_SEPARATOR = "\n\n"
# A conclusion: "the correct digit is 2", "Answer: **1**", "I choose image 2", or "Image 2 is the
# correct answer". The last one in the reply counts, and a bare "1"/"2" reply counts too. Descriptions
# such as "Image 1 shows a teddy bear" are deliberately not matched, so a reply cut off before its
# conclusion has no stated answer. Only a diagnostic: the scored answer is always the forced choice.
_CONCLUSIONS = (
    re.compile(r"\b(?:answer|digit|choice|correct|pick|choose|chose|select)\w*\s*(?:is|would\s+be)?\s*[:=]?\s*"
               r"(?:\*\*)?\s*(?:the\s+)?(?:image|word|option|number)?\s*\(?\s*(?:\*\*)?([12])\b", re.IGNORECASE),
    re.compile(r"\b([12])\b\W{0,6}(?:is|would\s+be)\s+(?:the\s+)?(?:correct|right|answer)", re.IGNORECASE),
    re.compile(r"\b(?:therefore|thus|hence|so)\b[^.\n]{0,150}?\b(?:is|would\s+be)\s+(?:\*\*)?\s*(?:the\s+)?"
               r"(?:image|word|option|number)?\s*\(?\s*(?:\*\*)?([12])\b", re.IGNORECASE),
)


def stated_answer(reply):
    """The answer a free reply states, "1" or "2", or None if it states none."""
    text = (reply or "").strip().strip("*").rstrip(".").strip()
    if text in ("1", "2"):
        return text
    found = [(m.start(1), m.group(1)) for pattern in _CONCLUSIONS for m in pattern.finditer(reply or "")]
    return max(found)[1] if found else None


class ReasonThenChooseMixin:
    """Put before the evaluator class. The base supplies _generate(messages) -> (text, n_tokens,
    finished) and the standard score_choices / _call_api."""

    def _rtc_init(self):
        self._rtc_cached = None      # (messages object, reply) of the trial being scored
        self.trial_log = []          # one entry per score_choices call, in trial order

    def score_choices(self, messages, choices=("1", "2"), prefill="Answer: "):
        reply, n_tokens, finished = self._generate(messages)
        self._rtc_cached = (messages, reply)
        reasoned_prefill = (reply + ANSWER_SEPARATOR + prefill) if reply else prefill
        forced = super().score_choices(messages, choices=choices, prefill=reasoned_prefill)
        self.trial_log.append({"reasoning_tokens": n_tokens, "reasoning_finished": finished,
                               "stated_answer": stated_answer(reply)})
        return forced

    def _call_api(self, messages):
        cached = self._rtc_cached
        if cached is not None and cached[0] is messages:
            self._rtc_cached = None
            return cached[1]
        return super()._call_api(messages)


def reasoning_metrics(trials):
    n = len(trials)
    if not n or "reasoning_tokens" not in trials[0]:
        return {}
    stated = [t for t in trials if t.get("stated_answer") is not None]
    return {
        "mean_reasoning_tokens": sum(t["reasoning_tokens"] for t in trials) / n,
        "frac_reasoning_cut_off": sum(not t["reasoning_finished"] for t in trials) / n,
        "frac_stated_answer": len(stated) / n,
        # Of the trials whose reply states an answer, how often the forced choice read after it agrees.
        "stated_forced_agreement": (sum(str(t["stated_answer"]) == str(t.get("choice", t["reported"]))
                                        for t in stated) / len(stated)) if stated else None,
    }


def wrap_adapter(adapter):
    """The task adapter, with each trial's reasoning stats attached after the task has run."""

    class Wrapped:
        NAME = adapter.NAME

        def __getattr__(self, name):
            return getattr(adapter, name)

        @staticmethod
        def run(evaluator, condition, seed):
            log = getattr(evaluator, "trial_log", None)
            if log is None:
                return adapter.run(evaluator, condition, seed)
            log.clear()
            result = adapter.run(evaluator, condition, seed)
            trials = result.get("trials") or []
            if trials:
                if len(log) != len(trials):
                    raise RuntimeError(f"{len(log)} scored choices for {len(trials)} trials; "
                                       "this task does not score exactly one choice per trial")
                for trial, extra in zip(trials, log):
                    trial.update(extra)
                result["metrics"].update(reasoning_metrics(trials))
            return result

    return Wrapped()


def build_evaluator(model_cfg):
    """bench.models.build_evaluator, but `scoring: reason_then_choose` selects the reasoning evaluator."""
    from bench import models

    scoring = model_cfg.get("scoring", "standard")
    if scoring == "standard":
        return models.build_evaluator(model_cfg)
    if scoring != "reason_then_choose":
        raise ValueError(f"unknown scoring {scoring!r}; expected 'standard' or 'reason_then_choose'")

    import torch

    class ReasonThenChooseEvaluator(ReasonThenChooseMixin, models.Qwen35Evaluator):
        def __init__(self, **kwargs):
            super().__init__(**kwargs)
            self._rtc_init()

        def _generate(self, messages):
            """Qwen35Evaluator._call_api, also returning the token count and whether it ended by itself."""
            inputs = self._prepare_inputs(messages)
            with torch.no_grad():
                out = self.model.generate(
                    **inputs, max_new_tokens=self.max_new_tokens, do_sample=False,
                    eos_token_id=self.eos_token_id, pad_token_id=self.pad_token_id,
                )
            new = out[0, inputs.input_ids.shape[1]:]
            n_tokens = int(new.shape[0])
            finished = n_tokens < self.max_new_tokens or (n_tokens > 0 and int(new[-1]) in self.eos_token_id)
            reply = self.processor.batch_decode(new.unsqueeze(0), skip_special_tokens=True)[0]
            return reply.strip(), n_tokens, finished

    evaluator = ReasonThenChooseEvaluator(
        model_id=model_cfg["model_id"],
        max_image_size=model_cfg.get("max_image_size", 512),
        max_new_tokens=model_cfg.get("max_new_tokens", 100),
        layer_variant=model_cfg.get("layer_variant"),
        name=model_cfg["name"],
        device_map=model_cfg.get("device_map", "cuda"),
    )
    print(f"Scoring {model_cfg['name']} as reason_then_choose: reply up to {evaluator.max_new_tokens} tokens, "
          "then the forced choice with the reply in context", flush=True)
    return evaluator


def install():
    import bench.runner
    from bench.registry import get_adapter

    bench.runner.build_evaluator = build_evaluator
    bench.runner.get_adapter = lambda name: wrap_adapter(get_adapter(name))


if __name__ == "__main__":
    install()
    import run_bench

    run_bench.main()


In [ ]:
%%writefile {REPO}/runpod/assoc_preflight.py
#!/usr/bin/env python
"""Check the associative-inference preflights and estimate the full runs.

Usage (from the repo root):
    python runpod/assoc_preflight.py --check <preflight config>=<full config> [...]
        [--capacity GB] [--price DOLLARS_PER_HOUR] [--json out.json]

For each model in each preflight config: every preflight cell exists with no error or skip, the forced
choice holds real probability mass (mean choice_mass >= 0.8), the free replies are readable text (a
digit, or English sentences, not token soup), and peak GPU memory stays under 95% of --capacity.
Reason-then-choose cells must also carry their reasoning stats (proof they ran through
run_bench_ext.py). Prints PASS/FAIL per model with the reasons, some replies, and a rough time for
that model's part of the full config: (its trials) x (seconds per trial at the preflight's size),
an upper bound since most full cells are smaller. --json writes {model: {"passed", "reasons"}}.
"""
import argparse
import json
import re
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))

from bench.config import expand_runs, load_config  # noqa: E402

COMMON = {"the", "a", "an", "is", "are", "was", "of", "and", "to", "in", "with", "image", "images", "cue", "pair",
          "pairs", "paired", "first", "second", "which", "this", "that", "it", "be", "on", "for", "as", "word",
          "words", "option", "answer", "correct", "therefore", "both", "not", "so", "based", "shows", "associated"}


def readable(reply):
    reply = (reply or "").strip()
    if reply.strip("*").rstrip(".") in ("1", "2") or re.fullmatch(r"\**[12]\**[.):]?\s+\**[A-Za-z][A-Za-z '-]*\**\.?", reply):
        return True                                       # "2" or "2. BLOUSE"
    if not reply:
        return False
    ascii_share = sum(ch.isascii() for ch in reply) / len(reply)
    words = re.findall(r"[A-Za-z]{1,}", reply.lower())
    common = sum(w in COMMON for w in words) / len(words) if words else 0
    return ascii_share >= 0.9 and len(words) >= 3 and common >= 0.15


def results(cfg):
    out = {}
    for run in expand_runs(cfg):
        path = Path(cfg["output_dir"]) / cfg["run_name"] / run.task / run.result_name
        out.setdefault(run.model["name"], []).append((run, json.loads(path.read_text()) if path.exists() else None))
    return out


def full_trials(cfg):
    trials = {}
    for run in expand_runs(cfg):
        trials[run.model["name"]] = trials.get(run.model["name"], 0) + run.condition.get("n_trials", 10)
    return trials


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--check", nargs="+", required=True, metavar="PREFLIGHT=FULL")
    parser.add_argument("--capacity", type=float, default=0.0, help="GB of GPU memory the jobs run on")
    parser.add_argument("--price", type=float, default=0.0)
    parser.add_argument("--json")
    args = parser.parse_args()

    verdicts, total_h = {}, 0.0
    for spec in args.check:
        pre_path, _, full_path = spec.partition("=")
        pre_cfg = load_config(pre_path)
        full = full_trials(load_config(full_path)) if full_path else {}
        print(f"===== {pre_path}" + (f"  (full run: {full_path})" if full_path else ""))
        for model, cells in results(pre_cfg).items():
            reasons, replies, ok_replies, masses, per_trial, peaks = [], [], 0, [], [], []
            rtc = pre_cfg_model(pre_cfg, model).get("scoring") == "reason_then_choose"
            for run, d in cells:
                c = run.condition
                label = f"{c['pair_type']} {c['probe']} {c['n_chains']} chains {c['prompt']}"
                if d is None:
                    reasons.append(f"{label}: no result file")
                    continue
                if d.get("error"):
                    reasons.append(f"{label}: {d['error'].strip().splitlines()[-1][:160]}")
                    continue
                if d.get("skipped"):
                    reasons.append(f"{label}: skipped ({d['skipped']})")
                    continue
                trials = d.get("trials") or []
                if not trials:
                    reasons.append(f"{label}: no trials")
                    continue
                peaks.append(d.get("peak_gpu_gb", 0))
                per_trial.append(d.get("elapsed_s", 0) / len(trials))
                for t in trials:
                    masses.append(t.get("choice_mass", 0))
                    replies.append((label, t))
                    ok_replies += readable(t.get("free_response"))
                    if rtc and "reasoning_tokens" not in t:
                        reasons.append(f"{label}: no reasoning stats, so it did not run through run_bench_ext.py")
                        break
            if masses and sum(masses) / len(masses) < 0.8:
                reasons.append(f"mean choice_mass {sum(masses) / len(masses):.2f} is under 0.8: "
                               "the answer digits are not where the probability is")
            if replies and ok_replies < len(replies) - 1:
                reasons.append(f"only {ok_replies} of {len(replies)} replies read as text or a digit (possible gibberish)")
            if args.capacity and peaks and max(peaks) > 0.95 * args.capacity:
                reasons.append(f"peak {max(peaks)} GB is within 5% of the {args.capacity:.0f} GB available")
            passed = not reasons
            verdicts[model] = {"passed": passed, "reasons": reasons}

            peak = f"peak {max(peaks):.1f} GB" if peaks else "no peak recorded"
            line = f"{model:12s} {'PASS' if passed else 'FAIL'}   {peak}"
            if per_trial and model in full:
                secs = min(per_trial)
                hours = full[model] * secs / 3600
                total_h += hours
                line += f"   ~{secs:.1f} s/trial -> full run ~{hours:.1f} h at most ({full[model]} trials)"
            print(line)
            for r in reasons:
                print("     -", r)
            for label, t in replies[:3]:
                extra = (f" tokens={t['reasoning_tokens']} finished={t['reasoning_finished']} "
                         f"stated={t.get('stated_answer')}") if "reasoning_tokens" in t else ""
                print(f"     [{label}] target={t['target']} choice={t['reported']} mass={t.get('choice_mass', 0):.3f}{extra}")
                print(f"        reply: {(t.get('free_response') or '')[:300]!r}")
        print()
    if total_h:
        cost = f", about ${total_h * args.price:.0f} at ${args.price:.2f}/h" if args.price else ""
        print(f"Rough total for the models above: up to ~{total_h:.1f} h{cost}, plus model loading (a few minutes each).")
    if args.json:
        Path(args.json).write_text(json.dumps(verdicts, indent=2))


def pre_cfg_model(cfg, name):
    return next(m for m in cfg["models"] if m["name"] == name)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile {REPO}/runpod/assoc_gate.py
#!/usr/bin/env python
"""Apply the associative-inference gate written in configs/assoc_inf_ab.yaml before it was run.

Usage (from the repo root):
    python runpod/assoc_gate.py results/assoc_inf_ab [--models qwen3.5-2b,qwen3.5-4b] [--label TEXT]
                                [--json out.json] [--write-config configs/runpod/assoc_inf_full.yaml]

Gate (copied from the config header): a (layout, pair_type) is usable only if, for BOTH the 2B and
4B, at n_chains 2 and 5, direct_ab and direct_bc are above chance (95% Wilson lower bound > 0.5) and
neither is flagged BIAS (one answer on > 80% of trials). Only then is its inference curve read as
memory and a full sweep proposed. The prediction recorded with it: if image->image binding is the
failure, direct_ab fails for both pair types while direct_bc passes for `word`.

--models applies the same rule to another pair of models (e.g. the 9B and 27B), and the gate is also
reported for each model alone. A full-sweep config is only written for the 2B/4B gate, as specified.
"""
import argparse
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))

from bench.summarize import load_rows  # noqa: E402

IMAGES_PER_CHAIN = {"image": 4, "word": 3}
GATE_MODELS = ("qwen3.5-2b", "qwen3.5-4b")
GATE_CHAINS = (2, 5)
GATE_PROBES = ("direct_ab", "direct_bc")

# Full sweep for a passing layout: up to the dataset limit on Brady2008 (800 image chains, 1,200 word
# chains). The largest cells are 210k-250k tokens, so the 9B wants a 141 GB H200 there (its near-full-
# context peak is ~78 GB), and the 27B (about 190 GB near full context) needs two; it is left out.
FULL_LADDER = {"image": [2, 5, 25, 100, 250, 500, 800], "word": [2, 5, 25, 100, 250, 500, 800, 1200]}
FULL_MODELS = [("qwen3.5-0.8b", "Qwen/Qwen3.5-0.8B"), ("qwen3.5-2b", "Qwen/Qwen3.5-2B"),
               ("qwen3.5-4b", "Qwen/Qwen3.5-4B"), ("qwen3.5-9b", "Qwen/Qwen3.5-9B")]


def chains(row):
    return row["n_images"] // IMAGES_PER_CHAIN[row["pair_type"]]


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("run_dir")
    parser.add_argument("--json", help="write the gate result here")
    parser.add_argument("--write-config", help="if any layout passes, write a full-sweep config here")
    parser.add_argument("--models", default=",".join(GATE_MODELS), help="comma-separated model names")
    parser.add_argument("--label", default="", help="a heading for the printout, e.g. the scoring used")
    args = parser.parse_args()
    models = tuple(m.strip() for m in args.models.split(",") if m.strip())

    rows = [r for r in load_rows(args.run_dir) if r["task"] == "associative_inference"] \
        if Path(args.run_dir).exists() else []
    if not rows:
        print(f"No associative-inference results in {args.run_dir}.")
        return
    rows = [r for r in rows if r["model"] in models]
    if not rows:
        print(f"No associative-inference results for {', '.join(models)} in {args.run_dir}.")
        return
    index = {(r["model"], r["prompt"], r["pair_type"], r["probe"], chains(r)): r for r in rows}
    layouts = sorted({r["prompt"] for r in rows})
    pair_types = sorted({r["pair_type"] for r in rows})

    def ok(model, layout, pair_type, probe, n):
        r = index.get((model, layout, pair_type, probe, n))
        return r is not None and r["above_chance"] and "BIAS" not in r["flags"]

    results = []
    names = " and ".join(m.split("-")[-1].upper() for m in models)
    if args.label:
        print(f"== {args.label}")
    print("Gate: direct_ab and direct_bc above chance (Wilson lower bound > 0.5) with no BIAS flag,")
    print(f"for {'both the ' if len(models) > 1 else 'the '}{names}, at 2 and 5 chains.\n")
    for layout in layouts:
        for pair_type in pair_types:
            failures = []
            for model in models:
                for n in GATE_CHAINS:
                    for probe in GATE_PROBES:
                        r = index.get((model, layout, pair_type, probe, n))
                        if r is None:
                            failures.append(f"{model} {probe} {n}ch: missing")
                        elif not ok(model, layout, pair_type, probe, n):
                            why = "BIAS" if "BIAS" in r["flags"] else "at chance"
                            failures.append(f"{model} {probe} {n}ch: {r['accuracy']:.2f} "
                                            f"(low {r['ci95_low']:.2f}, top answer {r['most_common_frac']}) {why}")
            passed = not failures
            alone = {m: not any(f.startswith(m + " ") for f in failures) for m in models}
            results.append({"layout": layout, "pair_type": pair_type, "passed": passed, "failures": failures,
                            "passed_alone": alone})
            each = "   (alone: " + ", ".join(f"{m.split('-')[-1].upper()} {'pass' if ok_ else 'fail'}"
                                            for m, ok_ in alone.items()) + ")" if len(models) > 1 else ""
            print(f"{layout:9s} {pair_type:5s} {'PASS' if passed else 'FAIL'}{each}")
            for f in failures:
                print(f"      {f}")

    print("\nInference probe (read as memory only where the layout passed):")
    for layout in layouts:
        for pair_type in pair_types:
            cells = []
            for model in models:
                for n in sorted({chains(r) for r in rows if r["pair_type"] == pair_type}):
                    r = index.get((model, layout, pair_type, "inference", n))
                    if r:
                        bias = " BIAS" if "BIAS" in r["flags"] else ""
                        cells.append(f"{model.split('-')[-1]}@{n}: {r['accuracy']:.2f}{bias}")
            print(f"  {layout:9s} {pair_type:5s} " + "  ".join(cells))

    def direct(pair_type, probe):
        """Did this probe pass for both models at both gate sizes, under any layout?"""
        return any(all(ok(m, layout, pair_type, probe, n) for m in models for n in GATE_CHAINS)
                   for layout in layouts)

    if {"image", "word"} <= set(pair_types):
        predicted = (not direct("image", "direct_ab") and not direct("word", "direct_ab")
                     and direct("word", "direct_bc"))
        print("\nRecorded prediction (image->image binding is the failure: direct_ab fails for both pair "
              f"types, direct_bc passes for word): {'matches' if predicted else 'does not match'} the data.")

    passing = [r for r in results if r["passed"]]
    if args.json:
        Path(args.json).write_text(json.dumps({"run_dir": args.run_dir, "models": list(models), "label": args.label,
                                               "results": results, "passing": passing}, indent=2))
        print(f"wrote {args.json}")
    if passing and args.write_config and set(models) == set(GATE_MODELS):
        lines = ["# Full associative-inference sweep for the layout(s) that passed the gate",
                 f"# ({', '.join(p['layout'] + '/' + p['pair_type'] for p in passing)}), written by runpod/assoc_gate.py.",
                 "# The largest cells are 210k-250k tokens: the 9B wants a 141 GB H200 there. The 27B is left out",
                 "# (about 190 GB near full context); add it only on two H200s. Preflight before the full run.",
                 "", "run_name: assoc_inf_full", "", "output_dir: results", "", "seeds: [0, 1]", "", "models:"]
        for name, model_id in FULL_MODELS:
            lines += [f"  - name: {name}", f"    model_id: {model_id}", "    max_image_size: 512",
                      "    max_new_tokens: 100"]
        lines += ["", "tasks:"]
        for p in passing:
            lines += ["  - task: associative_inference", "    datasets: [Brady2008]",
                      f"    pair_types: [{p['pair_type']}]", "    probe_types: [inference, direct_ab, direct_bc]",
                      f"    n_chains: {FULL_LADDER[p['pair_type']]}", f"    prompts: [{p['layout']}]",
                      "    n_trials: 50", ""]
        Path(args.write_config).parent.mkdir(parents=True, exist_ok=True)
        Path(args.write_config).write_text("\n".join(lines))
        print(f"wrote {args.write_config}")
    elif not passing:
        print(f"\nNo layout passed for the {names}.")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile {REPO}/runpod/assoc_reasoned_report.py
#!/usr/bin/env python
"""Task 1 report: associative inference with standard vs reason-then-choose scoring, trial for trial.

Usage (from the repo root):
    python runpod/assoc_reasoned_report.py results/assoc_inf_ab results/assoc_inf_reasoned
        [--report out.md] [--csv out.csv] [--plot out.png]

Both runs use seed 0 and the same cells, so trial i of a cell sees the same episode and probe in
both. That is checked, not assumed: a cell is paired only if every trial's target and tested chain
match; otherwise it is compared unpaired and marked so.

Per cell: accuracy under each scoring, the change, an exact McNemar test on the discordant trials,
the BIAS flag (one answer on > 80% of trials) under each scoring, and for the reasoned run the mean
reply length, the share of replies cut off at the token limit, the share that state an answer, and
how often the forced choice agrees with the stated answer. Then the same pooled over cells, per model
and probe. Whether the reasoned run passes the repo's gate is assoc_gate.py's job (run it on the
reasoned directory); this report says whether and where the scoring changed the answers.
"""
import argparse
import csv
import json
import math
import sys
from collections import Counter, defaultdict
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))

from src.metrics import wilson_ci  # noqa: E402

BIAS_MAX_FRAC = 0.80
CUT_OFF_WARN = 0.20
PROBES = ("direct_ab", "direct_bc", "inference")
PROBE_STYLE = {"inference": ("#2a78d6", "o"), "direct_ab": ("#eb6834", "s"), "direct_bc": ("#1baf7a", "^")}
INK, MUTED, GRID = "#222222", "#777777", "#dddddd"


def load(run_dir):
    """{(model, pair_type, probe, n_chains, prompt): {(seed, trial): trial}}"""
    cells = defaultdict(dict)
    for path in sorted(Path(run_dir).glob("associative_inference/*.json")):
        d = json.loads(path.read_text())
        if d.get("error") or d.get("skipped"):
            continue
        c = d["run"]["condition"]
        key = (d["run"]["model"]["name"], c["pair_type"], c["probe"], c["n_chains"], c["prompt"])
        for t in d.get("trials", []):
            cells[key][(d["run"]["seed"], t["trial"])] = t
    return cells


def mcnemar_p(b, c):
    """Exact two-sided McNemar p: binomial test of the discordant split against 1/2."""
    n = b + c
    if n == 0:
        return 1.0
    k = min(b, c)
    tail = sum(math.comb(n, i) for i in range(k + 1)) / 2 ** n
    return min(1.0, 2 * tail)


def top_frac(trials):
    counts = Counter(str(t["reported"]) for t in trials)
    return counts.most_common(1)[0][1] / len(trials) if trials else 0.0


def bare(reply):
    return (reply or "").strip().rstrip(".") in ("1", "2")


def compare(plain, reasoned):
    """One row of numbers for a set of plain and reasoned trials (dicts keyed by (seed, trial))."""
    common = sorted(set(plain) & set(reasoned))
    paired = bool(common) and all(
        plain[k]["target"] == reasoned[k]["target"]
        and plain[k].get("metadata", {}).get("chain_index") == reasoned[k].get("metadata", {}).get("chain_index")
        for k in common)
    p_trials, r_trials = list(plain.values()), list(reasoned.values())
    row = {"paired": paired, "n_plain": len(p_trials), "n_reasoned": len(r_trials)}
    for name, trials in (("plain", p_trials), ("reasoned", r_trials)):
        k = sum(int(t["correct"]) for t in trials)
        acc, lo, hi = wilson_ci(k, len(trials)) if trials else (float("nan"),) * 3
        row.update({f"{name}_acc": acc, f"{name}_low": lo, f"{name}_high": hi,
                    f"{name}_top_frac": top_frac(trials),
                    f"{name}_bias": bool(trials) and top_frac(trials) > BIAS_MAX_FRAC})
    row["plain_long_replies"] = (sum(not bare(t.get("free_response")) for t in p_trials) / len(p_trials)
                                 if p_trials else None)
    if paired:
        b = sum(not plain[k]["correct"] and reasoned[k]["correct"] for k in common)
        c = sum(plain[k]["correct"] and not reasoned[k]["correct"] for k in common)
        row.update({"n_paired": len(common), "gained": b, "lost": c, "mcnemar_p": mcnemar_p(b, c),
                    "answer_changed": sum(plain[k]["reported"] != reasoned[k]["reported"] for k in common) / len(common)})
    else:
        row.update({"n_paired": 0, "gained": None, "lost": None, "mcnemar_p": None, "answer_changed": None})
    has_stats = [t for t in r_trials if "reasoning_tokens" in t]
    row["reasoning_recorded"] = len(has_stats) == len(r_trials) and bool(r_trials)
    if has_stats:
        stated = [t for t in has_stats if t.get("stated_answer") is not None]
        row.update({
            "mean_reasoning_tokens": sum(t["reasoning_tokens"] for t in has_stats) / len(has_stats),
            "cut_off": sum(not t["reasoning_finished"] for t in has_stats) / len(has_stats),
            "stated": len(stated) / len(has_stats),
            "stated_agree": (sum(str(t["stated_answer"]) == str(t["reported"]) for t in stated) / len(stated)
                             if stated else None),
        })
    return row


def fmt(v, spec=".2f"):
    if v is None or (isinstance(v, float) and math.isnan(v)):
        return "-"
    return format(v, spec)


def fmt_p(p):
    return "-" if p is None else ("<0.001" if p < 0.001 else f"{p:.3f}")


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("plain_dir")
    parser.add_argument("reasoned_dir")
    parser.add_argument("--report")
    parser.add_argument("--csv")
    parser.add_argument("--plot")
    args = parser.parse_args()

    plain, reasoned = load(args.plain_dir), load(args.reasoned_dir)
    keys = sorted(k for k in reasoned if k in plain)
    if not reasoned:
        print(f"No reasoned results in {args.reasoned_dir} yet.")
        return
    missing = sorted(k for k in reasoned if k not in plain)
    out = []

    def say(line=""):
        print(line)
        out.append(line)

    say("# Associative inference: standard vs reason-then-choose scoring")
    say()
    say(f"Standard: `{args.plain_dir}` (reply capped at 100 tokens, choice read with no reasoning in context). "
        f"Reasoned: `{args.reasoned_dir}` (reply up to 512 tokens, choice read after the reply). Same episodes, seed 0.")
    if missing:
        say()
        say(f"**{len(missing)} reasoned cells have no standard twin** (e.g. {' '.join(map(str, missing[0]))}). "
            "Upload the earlier results zip with results/assoc_inf_ab, or let the queue re-run configs/assoc_inf_ab.yaml.")
    if not keys:
        return

    rows = []
    for key in keys:
        model, pair_type, probe, n_chains, prompt = key
        rows.append({"model": model, "pair_type": pair_type, "probe": probe, "n_chains": n_chains,
                     "prompt": prompt, **compare(plain[key], reasoned[key])})

    unpaired = [r for r in rows if not r["paired"]]
    unrecorded = [r for r in rows if not r["reasoning_recorded"]]
    say()
    say(f"{len(rows)} cells compared; {len(rows) - len(unpaired)} paired trial for trial"
        + (f", **{len(unpaired)} unpaired** (episodes differ, compared as independent samples)" if unpaired else "")
        + ".")
    if unrecorded:
        say(f"**WARNING: {len(unrecorded)} reasoned cells carry no reasoning stats**, so they were not run with "
            "runpod/run_bench_ext.py and are standard scoring under another name. Do not use them.")

    # ---- pooled per model and probe ----
    say()
    say("## Pooled over layouts, pair types and chain counts")
    say()
    say("| model | probe | standard acc | reasoned acc | gained | lost | McNemar p | answers changed | "
        "reasoned BIAS cells | cut off | states answer | choice = stated |")
    say("|---|---|---|---|---|---|---|---|---|---|---|---|")
    pooled = []
    for model in sorted({r["model"] for r in rows}):
        for probe in PROBES:
            sub = [k for k in keys if k[0] == model and k[2] == probe]
            if not sub:
                continue
            p_all = {(k, t): v for k in sub for t, v in plain[k].items()}
            r_all = {(k, t): v for k in sub for t, v in reasoned[k].items()}
            row = compare(p_all, r_all)
            cell_rows = [r for r in rows if r["model"] == model and r["probe"] == probe]
            bias_cells = sum(r["reasoned_bias"] for r in cell_rows)
            pooled.append({"model": model, "probe": probe, **row})
            say(f"| {model} | {probe} | {fmt(row['plain_acc'])} | {fmt(row['reasoned_acc'])} "
                f"[{fmt(row['reasoned_low'])}, {fmt(row['reasoned_high'])}] | {fmt(row['gained'], 'd')} | "
                f"{fmt(row['lost'], 'd')} | {fmt_p(row['mcnemar_p'])} | {fmt(row['answer_changed'])} | "
                f"{bias_cells} of {len(cell_rows)} | {fmt(row.get('cut_off'))} | {fmt(row.get('stated'))} | "
                f"{fmt(row.get('stated_agree'))} |")

    # ---- reading ----
    say()
    say("## Reading")
    say()
    for r in pooled:
        if r["probe"] == "inference":
            continue
        verdict = ("above chance" if r["reasoned_low"] > 0.5 else "still at chance")
        change = ("no reliable change" if r["mcnemar_p"] is None or r["mcnemar_p"] >= 0.05
                  else "a reliable gain" if r["gained"] > r["lost"] else "a reliable loss")
        say(f"- {r['model']} {r['probe']}: {fmt(r['plain_acc'])} standard, {fmt(r['reasoned_acc'])} reasoned "
            f"({verdict} pooled; {change}, McNemar p {fmt_p(r['mcnemar_p'])}).")
    cut = [r for r in pooled if (r.get("cut_off") or 0) > CUT_OFF_WARN]
    if cut:
        say(f"- **{len(cut)} model/probe groups had over {CUT_OFF_WARN:.0%} of replies cut off at 512 tokens**, "
            "so the reasoning was often still unfinished: a longer limit is the next check before concluding.")
    else:
        say(f"- Replies mostly finished within 512 tokens (cut-off rate at most {CUT_OFF_WARN:.0%} everywhere), so "
            "truncation no longer explains the result.")
    say("- The gate itself (direct probes above chance with no BIAS at 2 and 5 chains, both models) is printed "
        "by assoc_gate.py on the reasoned run.")

    # ---- per cell ----
    say()
    say("## Every cell")
    say()
    say("| model | pair | probe | chains | layout | standard | reasoned | p | std top | rsn top | "
        "tokens | cut off | states | agree | paired |")
    say("|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|")
    for r in rows:
        say(f"| {r['model']} | {r['pair_type']} | {r['probe']} | {r['n_chains']} | {r['prompt']} | "
            f"{fmt(r['plain_acc'])} | {fmt(r['reasoned_acc'])} | {fmt_p(r['mcnemar_p'])} | "
            f"{fmt(r['plain_top_frac'])}{' B' if r['plain_bias'] else ''} | "
            f"{fmt(r['reasoned_top_frac'])}{' B' if r['reasoned_bias'] else ''} | "
            f"{fmt(r.get('mean_reasoning_tokens'), '.0f')} | {fmt(r.get('cut_off'))} | {fmt(r.get('stated'))} | "
            f"{fmt(r.get('stated_agree'))} | {'yes' if r['paired'] else 'NO'} |")
    say()
    say("top = share of the most common answer (B = BIAS, over 0.80). tokens = mean reasoned reply length. "
        "cut off = reply hit the 512-token limit. states = reply states an answer (\"the correct digit is 2\", "
        "or a bare digit). agree = forced choice matches the stated answer, of the replies that state one.")

    if args.report:
        Path(args.report).write_text("\n".join(out) + "\n")
        print(f"\nwrote {args.report}")
    if args.csv:
        fields = list(rows[0])
        with open(args.csv, "w", newline="") as f:
            w = csv.DictWriter(f, fieldnames=fields)
            w.writeheader()
            w.writerows(rows)
        print(f"wrote {args.csv}")
    if args.plot:
        plot(rows, Path(args.plot))
        print(f"wrote {args.plot}")


def plot(rows, path):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    models = sorted({r["model"] for r in rows}, key=lambda m: float(m.split("-")[-1].rstrip("b")))
    fig, axes = plt.subplots(1, len(models), figsize=(5.2 * len(models), 5.2), squeeze=False)
    for ax, model in zip(axes[0], models):
        for probe in PROBES:
            sub = [r for r in rows if r["model"] == model and r["probe"] == probe]
            if not sub:
                continue
            color, marker = PROBE_STYLE[probe]
            ax.scatter([100 * r["plain_acc"] for r in sub], [100 * r["reasoned_acc"] for r in sub],
                       s=64, color=color, marker=marker, edgecolors="white", linewidths=1.5, label=probe, zorder=3)
        ax.plot([0, 100], [0, 100], color=MUTED, lw=1, zorder=1)
        ax.axhline(50, ls="--", color="#999999", lw=1, zorder=1)
        ax.axvline(50, ls="--", color="#999999", lw=1, zorder=1)
        ax.set_xlim(0, 100)
        ax.set_ylim(0, 100)
        ax.set_aspect("equal")
        ax.grid(color=GRID, lw=0.6, zorder=0)
        for side in ("top", "right"):
            ax.spines[side].set_visible(False)
        ax.set_xlabel("Standard scoring accuracy (%)", color=INK)
        ax.set_ylabel("Reason-then-choose accuracy (%)", color=INK)
        ax.set_title(f"{model}: one point per cell", fontsize=11, color=INK)
        ax.text(98, 3, "above the diagonal = reasoning helped", ha="right", fontsize=8, color=MUTED)
        ax.legend(frameon=False, fontsize=9, loc="upper left")
    fig.suptitle("Associative inference, standard vs reason-then-choose (dashed = chance)", color=INK)
    fig.tight_layout()
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=150)
    plt.close(fig)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile {REPO}/runpod/assoc_replies.py
#!/usr/bin/env python
"""Are the associative-inference answers real? One row per model (and per layout), from the trials.

Usage (from the repo root): python runpod/assoc_replies.py <run_dir> [--models a,b] [--report out.md]

check_results.py reads a reply's first "1" or "2" as its answer, which suits bare-digit replies but
misreads reasoning ("Image 1 shows a teddy bear...") as a disagreement. Here instead:
  readable   replies that are a digit, "2. BLOUSE", or English text (not token soup); GIBBERISH? under 0.9
  bare       replies that are just the digit
  states     replies that state an answer ("the correct digit is 2"); agree = forced choice matches it
  mass       mean probability on the two answer digits; LOW_MASS cells have over 10% of trials under 0.9
  BIAS cells one answer on over 80% of trials
"""
import argparse
import json
import sys
from collections import defaultdict
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent))

from assoc_preflight import readable  # noqa: E402
from run_bench_ext import stated_answer  # noqa: E402


def summarize(cells):
    trials = [t for c in cells for t in c["trials"]]
    n = len(trials)
    if not n:
        return None
    stated = [(t, stated_answer(t.get("free_response"))) for t in trials]
    stated = [(t, s) for t, s in stated if s is not None]
    low_mass = bias = 0
    for c in cells:
        ts = c["trials"]
        if ts and sum(t.get("choice_mass", 0) < 0.9 for t in ts) / len(ts) > 0.10:
            low_mass += 1
        counts = defaultdict(int)
        for t in ts:
            counts[t["reported"]] += 1
        if ts and max(counts.values()) / len(ts) > 0.80:
            bias += 1
    return {
        "cells": len(cells), "trials": n,
        "readable": sum(readable(t.get("free_response")) for t in trials) / n,
        "bare": sum((t.get("free_response") or "").strip().rstrip(".") in ("1", "2") for t in trials) / n,
        "states": len(stated) / n,
        "agree": sum(str(s) == str(t["reported"]) for t, s in stated) / len(stated) if stated else None,
        "mass": sum(t.get("choice_mass", 0) for t in trials) / n,
        "low_mass_cells": low_mass, "bias_cells": bias,
    }


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("run_dir")
    parser.add_argument("--models")
    parser.add_argument("--report")
    args = parser.parse_args()
    keep = set(args.models.split(",")) if args.models else None

    groups, errors = defaultdict(list), defaultdict(list)
    for path in sorted(Path(args.run_dir).glob("associative_inference/*.json")):
        d = json.loads(path.read_text())
        model = d["run"]["model"]["name"]
        if keep and model not in keep:
            continue
        if d.get("error") or d.get("skipped"):
            errors[model].append(path.name)
            continue
        cell = {"trials": d.get("trials") or []}
        groups[(model, "all")].append(cell)
        groups[(model, d["run"]["condition"]["prompt"])].append(cell)

    out = []

    def say(line=""):
        print(line)
        out.append(line)

    say(f"Replies in {args.run_dir}")
    say()
    say("| model | layout | cells | readable | bare digit | states answer | choice = stated | mass | LOW_MASS cells | BIAS cells | flags |")
    say("|---|---|---|---|---|---|---|---|---|---|---|")
    order = sorted(groups, key=lambda k: (float(k[0].split("-")[-1].rstrip("b")), k[1] != "all", k[1]))
    for key in order:
        s = summarize(groups[key])
        if s is None:
            continue
        flags = []
        if s["readable"] < 0.9:
            flags.append("GIBBERISH?")
        if key[1] == "all" and errors.get(key[0]):
            flags.append(f"{len(errors[key[0]])} ERROR/SKIPPED")
        agree = "-" if s["agree"] is None else f"{s['agree']:.2f}"
        say(f"| {key[0]} | {key[1]} | {s['cells']} | {s['readable']:.2f} | {s['bare']:.2f} | {s['states']:.2f} | "
            f"{agree} | {s['mass']:.3f} | {s['low_mass_cells']} | {s['bias_cells']} | {' '.join(flags)} |")
    for model, names in errors.items():
        say()
        say(f"{model}: {len(names)} cells failed or were skipped, e.g. {names[0]}")
    if args.report:
        Path(args.report).write_text("\n".join(out) + "\n")
        print(f"\nwrote {args.report}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile {REPO}/runpod/run_queue.py
#!/usr/bin/env python
"""Run benchmark jobs one after another, detached from the notebook.

Usage: python runpod/run_queue.py <queue.json>

queue.json holds {"name", "python", "repo", "log_dir", "env", "jobs": [{"config", "task", "gpus", "entry"}]}.
Each job is `python <entry> <config> [--task <task>]` with CUDA_VISIBLE_DEVICES set to its GPUs, where
entry is run_bench.py unless the job names another (runpod/run_bench_ext.py for reason-then-choose);
a job without a task runs every task in its config, loading the model once.
The status file <log_dir>/<name>.status.json is rewritten after every change, and each job gets
its own log. A failed job does not stop the queue. The runner skips result files that already
exist, so starting the same queue again resumes where it stopped.
"""
import json
import os
import subprocess
import sys
from datetime import datetime
from pathlib import Path


def now():
    return datetime.now().isoformat(timespec="seconds")


def main():
    queue = json.loads(Path(sys.argv[1]).read_text())
    log_dir = Path(queue["log_dir"])
    log_dir.mkdir(parents=True, exist_ok=True)
    status_path = log_dir / f"{queue['name']}.status.json"
    status = {"name": queue["name"], "pid": os.getpid(), "started": now(), "finished": None,
              "jobs": [{**job, "state": "pending", "start": None, "end": None,
                        "returncode": None, "log": None} for job in queue["jobs"]]}

    def save():
        tmp = status_path.with_suffix(".tmp")
        tmp.write_text(json.dumps(status, indent=2))
        tmp.replace(status_path)

    save()
    for i, job in enumerate(status["jobs"]):
        log = log_dir / f"{queue['name']}_{i:02d}_{Path(job['config']).stem}_{job.get('task') or 'all'}.log"
        job.update(state="running", start=now(), log=str(log))
        save()
        env = {**os.environ, **queue.get("env", {}), "CUDA_VISIBLE_DEVICES": str(job.get("gpus", "0"))}
        cmd = [queue["python"], job.get("entry") or "run_bench.py", job["config"]]
        if job.get("task"):
            cmd += ["--task", job["task"]]
        cmd += job.get("extra_args", [])
        with open(log, "a") as f:
            f.write(f"# {now()} {' '.join(cmd)}  (CUDA_VISIBLE_DEVICES={env['CUDA_VISIBLE_DEVICES']})\n")
            f.flush()
            returncode = subprocess.call(cmd, cwd=queue["repo"], env=env, stdout=f, stderr=subprocess.STDOUT)
        job.update(state="done" if returncode == 0 else "failed", end=now(), returncode=returncode)
        save()
    status["finished"] = now()
    save()


if __name__ == "__main__":
    main()


In [ ]:
%%writefile {REPO}/runpod/progress.py
#!/usr/bin/env python
"""Done / total result files for each config, using the runner's own run expansion.

Usage (from the repo root): python runpod/progress.py [--json] <config.yaml>[:<task>] [...]

bench.runner imports torch, so its done-test is repeated here to keep this check fast and usable
while a job holds the GPU. A result counts as done exactly as in bench.runner.is_done: the file
exists, parses, has no error, and was not skipped. Running this also validates each config's sizes
against the datasets (expand_runs raises on a size the data cannot supply), without loading a model.
--json prints one JSON list ({config, task, done, failed, skipped, todo, total} per spec) instead.
"""
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))

from bench.config import expand_runs, load_config  # noqa: E402


def state(path):
    if not path.exists():
        return "todo"
    try:
        result = json.loads(path.read_text())
    except json.JSONDecodeError:
        return "todo"
    if result.get("error") is not None:
        return "failed"
    if result.get("skipped"):
        return "skipped"
    return "done"


def main():
    args = sys.argv[1:]
    as_json = "--json" in args
    rows = []
    for spec in (a for a in args if a != "--json"):
        config_path, _, task = spec.partition(":")
        cfg = load_config(config_path)
        if task:
            cfg["tasks"] = [t for t in cfg["tasks"] if t["task"] == task]
        runs = expand_runs(cfg)
        states = [state(Path(cfg["output_dir"]) / cfg["run_name"] / r.task / r.result_name) for r in runs]
        counts = {s: states.count(s) for s in ("done", "failed", "skipped", "todo")}
        rows.append({"config": config_path, "task": task or None, **counts, "total": len(runs)})
        if not as_json:
            label = config_path + (f" --task {task}" if task else "")
            print(f"{label:58s} {counts['done']:4d}/{len(runs):<4d} done   "
                  f"failed={counts['failed']} skipped={counts['skipped']} todo={counts['todo']}")
    if as_json:
        print(json.dumps(rows))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile {REPO}/runpod/plot_extra.py
#!/usr/bin/env python
"""Figures for the three tasks bench.plot does not draw: MST, color memory, associative inference.

Usage (from the repo root): python runpod/plot_extra.py <run_dir> [<run_dir> ...] [--out plots/runpod]

Same conventions as bench/plot.py: log-x study length, Wilson 95% CIs, one hue light -> dark by
model size, markers as a second cue. Adjacent blues in that ramp are hard to tell apart (the 4B and
9B pair fails a normal-vision separation check), so each line also carries a direct end label in
neutral ink. The 4B's published values (LIMITATIONS.md) are drawn as grey crosses for reference,
since the small-model result files live on the cluster, not on this pod.
"""
import argparse
import json
import sys
from collections import defaultdict
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402
from matplotlib.ticker import FuncFormatter, NullLocator  # noqa: E402

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))

from bench.summarize import load_rows  # noqa: E402

MODELS = {"qwen3.5-0.8b": ("#86b6ef", "o"), "qwen3.5-2b": ("#5598e7", "s"), "qwen3.5-4b": ("#2a78d6", "v"),
          "qwen3.5-9b": ("#1c5cab", "^"), "qwen3.5-27b": ("#0d366b", "D")}
INK, MUTED, GRID = "#222222", "#777777", "#dddddd"
REF_4B_ACC = {"mst": {1: 0.75, 5: 0.76, 25: 0.62, 100: 0.58, 250: 0.43, 576: 0.40},
              "named": {1: 0.86, 540: 0.38}}
REF_4B_LDI = {1: 0.35, 5: 0.29, 25: 0.00, 100: 0.21, 250: 0.14, 576: 0.23}
REF_4B_MAE = {1: 45.0}
PROBE_STYLE = {"inference": "-", "direct_ab": "--", "direct_bc": ":"}


def style(model):
    return MODELS.get(model, ("#888888", "x"))


def short(model):
    return model.split("-")[-1].upper()


def model_order(models):
    order = list(MODELS)
    return sorted(models, key=lambda m: order.index(m) if m in order else len(order))


def int_ticks(ax, values):
    """Ticks at the sizes actually run, labelled as plain numbers (log axes default to 10^k)."""
    values = sorted(set(values))
    ax.set_xticks(values)
    ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{int(round(v)):,}"))
    ax.xaxis.set_minor_locator(NullLocator())


def end_labels(ax, ends):
    """Direct labels at the line ends, nudged apart so two lines ending together stay readable.
    Call after the axis limits are final; `ends` is [(x, y, model)]."""
    if not ends:
        return
    lo, hi = ax.get_ylim()
    gap = 0.055 * (hi - lo)
    placed = []
    for x, y, model in sorted(ends, key=lambda e: e[1]):
        if placed and y - placed[-1][1] < gap:
            y = placed[-1][1] + gap
        placed.append((x, y, model))
    overflow = placed[-1][1] - hi
    if overflow > 0:
        placed = [(x, y - overflow, m) for x, y, m in placed]
    for x, y, model in placed:
        ax.annotate(short(model), (x, y), xytext=(7, 0), textcoords="offset points", va="center",
                    fontsize=9, color=INK, annotation_clip=False)


def frame(ax, xs, xlabel="Study sequence length"):
    ax.set_xscale("log")
    int_ticks(ax, xs)
    lo, hi = min(xs), max(xs)
    ax.set_xlim(lo / 1.4, hi * 1.9)           # room on the right for the end labels
    ax.set_xlabel(xlabel)
    ax.grid(color=GRID, lw=0.8)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)


def acc_axes(ax, rows, chance, ref=None, label_chance="Chance"):
    by_model = defaultdict(list)
    for r in rows:
        by_model[r["model"]].append(r)
    xs = [r["n_images"] for r in rows] + (list(ref) if ref else [])
    ends = []
    for model in model_order(by_model):
        rs = sorted(by_model[model], key=lambda r: r["n_images"])
        color, marker = style(model)
        x = [r["n_images"] for r in rs]
        y = [100 * r["accuracy"] for r in rs]
        err = [[100 * (r["accuracy"] - r["ci95_low"]) for r in rs], [100 * (r["ci95_high"] - r["accuracy"]) for r in rs]]
        ax.errorbar(x, y, yerr=err, color=color, marker=marker, ms=7, capsize=3, label=model, lw=2)
        ends.append((x[-1], y[-1], model))
    if ref:
        ax.plot(list(ref), [100 * v for v in ref.values()], "x", color=MUTED, ms=8, mew=2,
                label="Qwen3.5-4B (published)")
    ax.axhline(100 * chance, ls="--", color="#999999", lw=1, label=label_chance)
    frame(ax, xs)
    ax.set_ylim(0, 105)
    ax.set_ylabel("Accuracy (%)")
    end_labels(ax, ends)


def line_axes(ax, values, ylabel, chance=None, ref=None, chance_label="Chance", ylim=None):
    by_model = defaultdict(dict)
    for (model, n), v in values.items():
        by_model[model][n] = v
    ends = []
    for model in model_order(by_model):
        color, marker = style(model)
        ns = sorted(by_model[model])
        ys = [by_model[model][n] for n in ns]
        ax.plot(ns, ys, color=color, marker=marker, ms=7, lw=2, label=model)
        ends.append((ns[-1], ys[-1], model))
    if ref:
        ax.plot(list(ref), list(ref.values()), "x", color=MUTED, ms=8, mew=2, label="Qwen3.5-4B (published)")
    if chance is not None:
        ax.axhline(chance, ls="--", color="#999999", lw=1, label=chance_label)
    frame(ax, [n for (_, n) in values] + (list(ref) if ref else []))
    if ylim:
        ax.set_ylim(*ylim)
    ax.set_ylabel(ylabel)
    end_labels(ax, ends)


def save(fig, ax, path):
    handles, labels = ax.get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", ncol=3, frameon=True, bbox_to_anchor=(0.5, -0.02))
    fig.tight_layout(rect=(0, 0.12, 1, 1))
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print("wrote", path)


def per_cell_metric(run_dirs, task, mode, metric):
    """Trial-weighted mean of a per-file metric (ldi) or of a per-trial field (mae), per (model, n)."""
    sums = defaultdict(lambda: [0.0, 0])
    for path in sorted(p for d in run_dirs for p in Path(d).glob(f"{task}/*.json")):
        d = json.loads(path.read_text())
        cond = d["run"]["condition"]
        if mode and cond.get("mode") != mode:
            continue
        key = (d["run"]["model"]["name"], cond["n_images"])
        if metric == "ldi" and d.get("metrics") and "ldi" in d["metrics"] and d.get("trials"):
            sums[key][0] += d["metrics"]["ldi"] * len(d["trials"])
            sums[key][1] += len(d["trials"])
        elif metric == "mae":
            errors = [t["abs_error_degrees"] for t in d.get("trials", []) if t.get("abs_error_degrees") is not None]
            sums[key][0] += sum(errors)
            sums[key][1] += len(errors)
    return {k: s / n for k, (s, n) in sums.items() if n}


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("run_dirs", nargs="+")
    parser.add_argument("--out", default="plots/runpod")
    parser.add_argument("--models", help="comma-separated model names to draw (default: all)")
    parser.add_argument("--title", default="", help="prefix for the associative-inference figure title")
    args = parser.parse_args()
    dirs = [d for d in args.run_dirs if Path(d).exists()]
    if not dirs:
        print("No run directories found in", args.run_dirs)
        return
    out = Path(args.out)
    out.mkdir(parents=True, exist_ok=True)
    rows = load_rows(*dirs)
    if args.models:
        keep = {m.strip() for m in args.models.split(",")}
        rows = [r for r in rows if r["model"] in keep]

    mst = [r for r in rows if r["task"] == "mst"]
    if mst:
        fig, ax = plt.subplots(figsize=(7, 5))
        acc_axes(ax, mst, 1 / 3, REF_4B_ACC["mst"])
        ax.set_title("MST (old / similar / new), numbered layout\nAccuracy vs sequence length")
        save(fig, ax, out / "mst_accuracy.png")
        ldi = per_cell_metric(dirs, "mst", None, "ldi")
        if ldi:
            fig, ax = plt.subplots(figsize=(7, 5))
            line_axes(ax, ldi, "LDI = p(similar | lure) - p(similar | foil)", 0.0, REF_4B_LDI, "No discrimination")
            ax.set_title("MST lure discrimination index")
            save(fig, ax, out / "mst_ldi.png")

    named = [r for r in rows if r["task"] == "color_memory" and r["mode"] == "named"]
    if named:
        fig, ax = plt.subplots(figsize=(7, 5))
        acc_axes(ax, named, 1 / 6, REF_4B_ACC["named"])
        ax.set_title("Color memory, named (6 colors), numbered layout\nAccuracy vs sequence length")
        save(fig, ax, out / "color_named_accuracy.png")

    mae = per_cell_metric(dirs, "color_memory", "continuous", "mae")
    if mae:
        fig, ax = plt.subplots(figsize=(7, 5))
        line_axes(ax, mae, "Mean absolute error (degrees)", 90.0, REF_4B_MAE, "Chance (90 degrees)", ylim=(0, 180))
        ax.set_title("Color memory, continuous report\nError vs sequence length (lower is better)")
        save(fig, ax, out / "color_continuous_error.png")

    assoc = [r for r in rows if r["task"] == "associative_inference"]
    if assoc:
        layouts = sorted({r["prompt"] for r in assoc})
        pair_types = sorted({r["pair_type"] for r in assoc})
        per_chain = {"image": 4, "word": 3}
        fig, axes = plt.subplots(len(pair_types), len(layouts), figsize=(4.5 * len(layouts), 3.8 * len(pair_types)),
                                 squeeze=False, sharey=True)
        for i, pair_type in enumerate(pair_types):
            for j, layout in enumerate(layouts):
                ax = axes[i][j]
                cells = defaultdict(list)
                for r in assoc:
                    if r["pair_type"] == pair_type and r["prompt"] == layout:
                        cells[(r["model"], r["probe"])].append(r)
                xs = []
                for (model, probe), rs in sorted(cells.items()):
                    rs.sort(key=lambda r: r["n_images"])
                    color, marker = style(model)
                    x = [r["n_images"] // per_chain[pair_type] for r in rs]
                    xs += x
                    ax.plot(x, [100 * r["accuracy"] for r in rs], color=color, marker=marker, ms=6,
                            ls=PROBE_STYLE.get(probe, "-"), lw=1.8, label=f"{model} {probe}")
                ax.axhline(50, ls="--", color="#999999", lw=1)
                frame(ax, xs or [1], xlabel="Chains studied")
                ax.set_ylim(0, 105)
                ax.set_title(f"{layout} / {pair_type} pairs", fontsize=10)
            axes[i][0].set_ylabel("Accuracy (%)")
        if args.title:
            fig.suptitle(args.title, color=INK)
        save(fig, axes[0][0], out / "assoc_inference.png")


if __name__ == "__main__":
    main()


Validate every config against the datasets (no model is loaded):

In [ ]:
bench("runpod/progress.py", PREFLIGHT1, PREFLIGHT2, PLAIN_2B_4B, TASK1, TASK2)

## 8. Preflight (about 10–15 minutes)

Loads each model and runs two trials per cell at the largest cells it will see: the 2B and 4B with reason-then-choose at 5 chains, the 9B and 27B at 25 chains. Each model passes if every cell ran, the 1/2 digits hold most of the next-token probability, and the replies read as text or a digit (the 27B once produced gibberish on the cluster). It also prints peak GPU memory and a time and cost estimate. A model that fails is held back from the full runs.

In [ ]:
jobs = ([(PREFLIGHT1, GPUS, EXT)] if RUN_TASK1 else []) + ([(PREFLIGHT2, GPUS, None)] if RUN_TASK2 else [])
if jobs:
    launch("preflight", jobs)
    wait("preflight", every=20)
else:
    print("RUN_TASK1 and RUN_TASK2 are both off; nothing to do.")

In [ ]:
checks = ([f"{PREFLIGHT1}={TASK1}"] if RUN_TASK1 else []) + ([f"{PREFLIGHT2}={TASK2}"] if RUN_TASK2 else [])
verdict_file = OUT / "assoc_preflight.json"
verdict_file.unlink(missing_ok=True)
if checks:
    bench("runpod/assoc_preflight.py", "--check", *checks, "--capacity", round(capacity(), 1),
          "--price", PRICE_PER_HOUR, "--json", verdict_file, check=False)
    if RUN_TASK1 and (progress(PLAIN_2B_4B) or {}).get("todo"):
        print("(Plus about an hour to re-run the missing 2B/4B gate cells first; see section 3.)")
PREFLIGHT_OK = json.loads(verdict_file.read_text()) if verdict_file.exists() else {}

**How to read it.** `FAIL` with `OutOfMemoryError`: the model does not fit this GPU. `no reasoning stats`: a reason-then-choose cell did not run through `run_bench_ext.py`, so it would be mislabelled. Replies that are not text or a digit: do not run that model. After fixing something, re-run the two cells above (finished cells are kept).

## 9. Full runs (background)

One queue, in order: any missing 2B/4B gate cells (task 1's comparison), task 1 (2B and 4B, reason-then-choose), then task 2 (9B and 27B). A model that failed its preflight is left out (the gate needs both models of a pair, so that pair cannot be gated). You can close the browser; come back and run `status()`. To stop, use `stop("main")` below; launching again resumes.

In [ ]:
def passing(models):
    """The models whose preflight passed (all of them with FORCE_MAIN); says why the others are held back."""
    ok = []
    for m in models:
        if PREFLIGHT_OK.get(m, {}).get("passed") or FORCE_MAIN:
            ok.append(m)
            continue
        print(f"Holding back {m}: its preflight did not pass.")
        for r in PREFLIGHT_OK.get(m, {}).get("reasons", ["no preflight result"]):
            print("   -", r)
    return ok


jobs = []
if RUN_TASK1 and (ok := passing(TASK1_MODELS)):
    only = ok if len(ok) < len(TASK1_MODELS) else None
    jobs += [(PLAIN_2B_4B, GPUS, None, only), (TASK1, GPUS, EXT, only)]   # the first skips restored cells
if RUN_TASK2 and (ok := passing(TASK2_MODELS)):
    jobs.append((TASK2, GPUS, None, ok if len(ok) < len(TASK2_MODELS) else None))
if jobs:
    launch("main", jobs)
else:
    print("Nothing to launch.")

In [ ]:
status("main")

In [ ]:
wait("main", every=300)     # blocks until everything finishes; interrupt any time, the jobs keep running

In [ ]:
# Uncomment to stop the full runs (finished cells are kept; launching again resumes):
# stop("main")

## 10. Task 1 results: 2B and 4B, reason-then-choose

First the repo's gate, applied to the reasoned run exactly as to the original one. Then every reasoned trial against its twin from the gate run (same episode, same probe):

- **gained / lost**: trials that went wrong → right and right → wrong; McNemar's exact test asks whether that split is more than chance.
- **cut off**: replies that hit the 512-token limit. If many are cut off, 512 is not enough and the test is not finished.
- **states answer / choice = stated**: how often the reply ends in an answer ("the correct digit is 2"), and how often the choice read afterwards matches it. High agreement means the forced choice is reading the model's own conclusion.

**How to read it.** If the direct probes now pass the gate, the original failure was the answer being read too early, and the inference probe becomes interpretable. If they stay at chance with most replies finished, the models genuinely cannot retrieve the studied pairs in this format.

In [ ]:
if (REPO / "results/assoc_inf_reasoned").exists():
    bench("runpod/assoc_gate.py", "results/assoc_inf_reasoned", "--models", ",".join(TASK1_MODELS),
          "--label", "Task 1: 2B and 4B, reason-then-choose scoring", "--json", OUT / "assoc_gate_task1.json", check=False)
else:
    print("No task 1 results yet.")

In [ ]:
if (REPO / "results/assoc_inf_reasoned").exists():
    bench("runpod/assoc_reasoned_report.py", "results/assoc_inf_ab", "results/assoc_inf_reasoned",
          "--report", OUT / "TASK1_REASONED_REPORT.md", "--csv", OUT / "task1_reasoned_cells.csv",
          "--plot", "plots/runpod_assoc/task1_standard_vs_reasoned.png", check=False)
    show_replies("results/assoc_inf_reasoned", k=1)

## 11. Task 2 results: 9B and 27B, the repo's gate

The gate for the 9B and 27B as a pair (the same rule as for the 2B and 4B), and for each alone. Then the reply checks, on all four models for comparison, one row per model and layout:

| Column | Meaning |
|---|---|
| readable | replies that are a digit or English text, not token soup. Under 0.9 is flagged `GIBBERISH?` |
| bare digit | replies that are just "1" or "2", as the prompt asks. The rest are reasoning, often cut off at 100 tokens |
| states answer / choice = stated | replies that end in an answer, and how often the forced choice matches it |
| mass | probability on the two answer digits; `LOW_MASS` cells have over 10% of trials under 0.9 |
| BIAS cells | one answer on over 80% of trials |

In [ ]:
if any((REPO / "results/assoc_inf_ab/associative_inference").glob("qwen3.5-9b__*")) or \
        any((REPO / "results/assoc_inf_ab/associative_inference").glob("qwen3.5-27b__*")):
    bench("runpod/assoc_gate.py", "results/assoc_inf_ab", "--models", ",".join(TASK2_MODELS),
          "--label", "Task 2: 9B and 27B, standard scoring", "--json", OUT / "assoc_gate_task2.json", check=False)
else:
    print("No task 2 results yet.")

In [ ]:
if (REPO / "results/assoc_inf_ab").exists():
    bench("runpod/assoc_replies.py", "results/assoc_inf_ab", "--report", OUT / "TASK2_REPLIES.md", check=False)
    show_replies("results/assoc_inf_ab", models=TASK2_MODELS, k=1)

## 12. Figures

In [ ]:
fig_dir = REPO / "plots/runpod_assoc"
if (REPO / "results/assoc_inf_ab").exists():
    bench("runpod/plot_extra.py", "results/assoc_inf_ab", "--models", ",".join(TASK2_MODELS),
          "--out", "plots/runpod_assoc/task2_9b_27b", "--title", "Task 2: 9B and 27B, standard scoring", check=False)
if (REPO / "results/assoc_inf_reasoned").exists():
    bench("runpod/plot_extra.py", "results/assoc_inf_reasoned", "--out", "plots/runpod_assoc/task1_reasoned",
          "--title", "Task 1: 2B and 4B, reason-then-choose scoring", check=False)
for png in sorted(fig_dir.rglob("*.png")):
    display(Markdown(f"**{png.relative_to(fig_dir)}**"))
    display(Image(filename=str(png)))

## 13. Package

One zip in `/workspace/outputs`; download it (right-click, Download). Unzipped at the repo root it adds `results/assoc_inf_ab/` (the 9B and 27B beside the 2B and 4B), `results/assoc_inf_reasoned/`, the configs and scripts, and `plots/runpod_assoc/`. Reports, preflights and logs go to `runpod_extras/`. It also holds every result restored in section 3, so it replaces the earlier zip.

In [ ]:
stamp = datetime.now().strftime("%Y%m%d_%H%M")
bundle = OUT / f"qwen35_runpod_results_assoc_{stamp}.zip"
freeze = OUT / "pip_freeze.txt"
LIST_PACKAGES = ("import importlib.metadata as m; print(chr(10).join(sorted("
                 "str(d.metadata['Name']) + '==' + d.version for d in m.distributions())))")
r = subprocess.run([str(PY), "-c", LIST_PACKAGES], capture_output=True, text=True)
if r.returncode == 0:
    freeze.write_text(r.stdout)

into_repo = ["results/assoc_inf_ab", "results/assoc_inf_reasoned", "configs/runpod", "runpod", "plots/runpod_assoc"]
restored_elsewhere = sorted(p for p in (REPO / "results").glob("*") if p.is_dir()
                            and p.name not in ("assoc_inf_ab", "assoc_inf_reasoned")
                            and not p.name.startswith("runpod_preflight"))
extras = {f"results/{p}": f"runpod_extras/results/{p}"
          for p in ("runpod_preflight_assoc", "runpod_preflight_assoc_reasoned")}
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as z:
    for rel in into_repo + [p.relative_to(REPO).as_posix() for p in restored_elsewhere]:
        for f in sorted((REPO / rel).rglob("*")):
            if f.is_file() and "__pycache__" not in f.parts:
                z.write(f, f.relative_to(REPO).as_posix())
    for rel, dest in extras.items():
        for f in sorted((REPO / rel).rglob("*")):
            if f.is_file():
                z.write(f, f"{dest}/{f.relative_to(REPO / rel).as_posix()}")
    for f in (OUT / "TASK1_REASONED_REPORT.md", OUT / "task1_reasoned_cells.csv", OUT / "TASK2_REPLIES.md", OUT / "assoc_gate_task1.json", OUT / "assoc_gate_task2.json",
              OUT / "assoc_preflight.json", freeze, REPO / "RUNPOD_SOURCE.json"):
        if f.exists():
            z.write(f, f"runpod_extras/{f.name}")
    for f in sorted(LOGS.glob("*")):
        if f.is_file():
            z.write(f, f"runpod_extras/logs/{f.name}")

with zipfile.ZipFile(bundle) as z:
    names = z.namelist()
groups = {}
for n in names:
    parts = n.split("/")
    key = "/".join(parts[:2] if parts[0] in ("results", "runpod_extras", "plots") else parts[:1])
    groups[key] = groups.get(key, 0) + 1
print(f"{bundle}  ({bundle.stat().st_size / 1e6:.1f} MB, {len(names)} files)")
for key, count in sorted(groups.items()):
    print(f"  {key:45s} {count:5d} files")

## Appendix. Stop the pod

Download the zip first. **Stop** keeps the `/workspace` volume and frees the GPU; **Terminate** deletes everything not on a network volume.

In [ ]:
pod = os.environ.get("RUNPOD_POD_ID")
print("Pod id:", pod or "unknown (not on RunPod?)")
# run(["runpodctl", "stop", "pod", pod])

## Troubleshooting

- **"CUDA driver version is insufficient"**: the pod's driver is older than 570. Redeploy with CUDA 12.8 or newer.
- **`OutOfMemoryError`** in the preflight: the model does not fit; on an H200 this should not happen here.
- **The first trial is slow**: the linear-attention kernels compile on first use (20–140 s), then are cached in `/workspace/.triton`.
- **Everything is several times slower than the estimate and the GPU draws little power**: the jobs are starved on the CPU. Set `CPU_THREADS = 8` and launch again.
- **The pod restarted**: run the settings and helpers cells, then `status("main")`; if the queue is gone, run the launch cell again.
- **The pod is gone**: start a new pod, upload the notebook, the repo zip and your latest results zip, and Run All. Finished cells are restored and skipped.
- **A job's full log**: the file named by `status()`, in `/workspace/runpod_logs`.